# 短期记忆模块
数据结构：字典列表
字典：执行和反思的记录


In [ ]:
from typing import Dict, List, Optional

class Memory:
    def __init__(self):
        self.records: List[Dict[str, str]] = []

    def add_record(self, record_type: str, content: str):
        """
        添加一条新的记录到记忆中。
        """
        self.records.append({"type": record_type, "content": content})
        print(f"📝 记忆已更新，新增一条 '{record_type}' 记录。\n")

    def get_trajectory(self) -> str:
        """
        获取完整的执行轨迹，包括每轮的尝试和评审员的反馈。
        """
        trajectory_parts = []
        for record in self.records:
            if record["type"] == "execution":
                trajectory_parts.append(f"---上一轮尝试---\n{record['content']}")
            elif record["type"] == "reflection":
                trajectory_parts.append(f"---评审员反馈---\n{record['content']}")
        return "\n\n".join(trajectory_parts)

    def get_last_execution(self) -> Optional[str]:
        """
        获取最近一次执行记录的内容。
        """
        for record in reversed(self.records):
            if record["type"] == "execution":
                return record["content"]
        return None


In [14]:
INITIAL_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。请根据以下要求，编写一个Python函数。
你的的代码必须包含完整的函数定义，包括函数名、参数列表、返回值类型（如果有）、函数体。遵循Python的PEP 8编码规范。

要求：{task}

请直接输出代码，不要包含如何额外的解释。
"""

In [15]:
REFLECT_PROMPT_TEMPLATE = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码的性能有极致的要求。
你的任务是审查以下Python代码，并专注于找出在<strong>算法效率</strong>上的主要瓶颈。

# 原始任务:
{task}

# 待审查代码:
```python
{code}
```

请分析代码的时间复杂度，并思考是否存在一种<strong>算法上更优</strong>的解决方案。
如果存在，请清晰地指出当前算法的不足，并提出具体的、可行的改进算法建议。
如果代码在算法层面已经达到最优，才能回答“无需改进”。

请直接输出你的审查结果，不要包含任何额外的解释和代码。
"""

In [16]:
REFINE_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。你的任务是根据评审员的反馈优化代码。

# 原始任务:
{task}

# 你上一轮尝试的代码:
{last_code_attempt}
评审员的反馈:
{feedback}

请根据评审员的反馈，生成一个优化后的新版本代码。
你的代码必须包含完整的函数定义和说明，并遵循Python的PEP 8编码规范。
请直接输出优化后的代码，不要包含任何额外的解释。
"""

In [ ]:
from dotenv import load_dotenv
from AgentsClient import HelloAgentsLLM

class ReflectionAgent:
    def __init__(self, llm_client: HelloAgentsLLM, max_iterations=3):
        self.llm_client = llm_client
        self.memory = Memory()
        self.max_iterations = max_iterations

    def run(self, task: str):
        initial_prompt = INITIAL_PROMPT_TEMPLATE.format(task=task)
        initial_code = self._get_llm_response(initial_prompt)
        self.memory.add_record("execution", initial_code)

        for i in range(self.max_iterations):
            # 评审员 对代码进行评审
            last_code = self.memory.get_last_execution()
            reflect_prompt = REFLECT_PROMPT_TEMPLATE.format(
                task=task,
                code=last_code
            )
            feedback = self._get_llm_response(reflect_prompt)
            self.memory.add_record("reflection", feedback)

            if "无需改进" in feedback:
                break

            # 程序员 优化代码
            refine_prompt = REFINE_PROMPT_TEMPLATE.format(
                task=task,
                last_code_attempt=last_code,
                feedback=feedback
            )
            refine_code = self._get_llm_response(refine_prompt)
            self.memory.add_record("execution", refine_code)
        
        # 任务完成，返回结果
        final = self.memory.get_last_execution()
        print(f"---任务执行轨迹---\n{self.memory.get_trajectory()}"
        + "\n------\n\n")
        return final

    def _get_llm_response(self, prompt: str) -> str:
        """
        调用LLM模型获取响应
        """
        messages = [{"role": "user", "content": prompt}]
        return self.llm_client.think(messages)

if __name__ == "__main__":
    # 加载 .env 文件中的环境变量
    load_dotenv(override=True)
    llm_client = HelloAgentsLLM()
    agent = ReflectionAgent(llm_client, max_iterations=10)
    task = "编写一个简单的Python函数，找出1到n之间所有的素数。"
    result = agent.run(task)
    print(f"任务执行结果:\n{result}")

正在调用 LLM 模型:  qwen-max
LLM 模型调用成功，开始接收流式响应...
```python
def find_primes(n: int) -> list:
    """
    返回1到n之间所有的素数列表。
    :param n: 整数n
    :return: 素数列表
    """
    if n < 2:
        return []
    
    primes = []
    for num in range(2, n + 1):
        is_prime = True
        for i in range(2, int(num ** 0.5) + 1):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    return primes
```
LLM 流式响应结束

📝 记忆已更新，新增一条 'execution' 记录。

正在调用 LLM 模型:  qwen-max
LLM 模型调用成功，开始接收流式响应...
当前算法的时间复杂度为O(n√n)。存在更优的解决方案。

当前算法逐个检查每个数字是否为素数，对于每个数字都执行了从2到该数字平方根的除法操作来验证其是否为素数，这导致了大量的重复计算。

改进建议采用埃拉托斯特尼筛法（Sieve of Eratosthenes），此方法通过标记非素数的方式有效地减少了不必要的计算，从而提高了效率。使用这种方法可以将时间复杂度降低至O(n log(log n))。
LLM 流式响应结束

📝 记忆已更新，新增一条 'reflection' 记录。

正在调用 LLM 模型:  qwen-max
LLM 模型调用成功，开始接收流式响应...
```python
def find_primes(n: int) -> list:
    """
    使用埃拉托斯特尼筛法返回1到n之间所有的素数列表。
    :param n: 整数n
    :return: 素数列表
    """
    if n < 2:
      

In [18]:
if "无须改进" in "时间复杂度为O(n log log n)，在理论上已经相当高效。对于找出1到n之间所有素数的问题，在算法层面上已经达到最优，无需改进。":
    print("123")